In [3]:
import pandas as pd
from pathlib import Path
import re
import numpy as np


# ============================================================
#                       CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# 1. MAIN PENTACAM DATA FOLDER
# ------------------------------------------------------------
#
# Example:
#
# Pentacam_CSV_Extracted/
# ├── Subfolder_1/
# │   ├── School_A/
# │   │   ├── 2025-09-04/
# │   │   │   ├── INDEX-LOAD_04-09-2025.csv
# │   │   │   ├── PACHY-LOAD_04-09-2025.csv
# │   │   │   └── ...
# │   │   └── 2025-09-05/
# │   └── School_B/
# ├── Subfolder_2/
# └── Subfolder_3/
#

MAIN_FOLDER = Path(
    r"C:\xx"
)


# ------------------------------------------------------------
# 2. MAPPING EXCEL FILE
# ------------------------------------------------------------

MAPPING_FILE = Path(
    r"Pentacam_PARAMETER.xlsx"
)


# ------------------------------------------------------------
# 3. MAPPING SHEET
# ------------------------------------------------------------

MAPPING_SHEET = "Full Re-check"


# ------------------------------------------------------------
# 4. OUTPUT FILE NAME
# ------------------------------------------------------------

OUTPUT_FILE = "merged_csv.xlsx"


# ------------------------------------------------------------
# 5. VALIDATION REPORT
# ------------------------------------------------------------

VALIDATION_FILE = "merge_validation_report.xlsx"


# ============================================================
#                  IDENTIFIER COLUMNS
# ============================================================

# These columns will appear ONLY ONCE in merged Excel.

IDENTIFIER_COLUMNS = [
    "Last Name:",
    "First Name:",
    "Pat-ID:",
    "D.o.Birth:",
    "Exam Eye:"
]


# ============================================================
#       PRIMARY MATCHING IDENTIFIERS
# ============================================================

# We use these for the actual merge.
#
# Names are NOT used as primary keys because names can have
# spelling/capitalization/spacing differences.
#
# Last Name and First Name will still be checked for
# consistency.

PRIMARY_MATCH_COLUMNS = [
    "Pat-ID:",
    "D.o.Birth:",
    "Exam Eye:"
]


# ============================================================
#             GENERAL COLUMN NORMALIZATION
# ============================================================

def normalize_column_name(column):
    """
    Standardize column names for comparison.

    Example:
        ' Pat-ID: ' -> 'Pat-ID:'
    """

    column = str(column)

    column = column.strip()

    column = re.sub(
        r"\s+",
        " ",
        column
    )

    return column


# ============================================================
#               NORMALIZE FILE NAME
# ============================================================

def normalize_file_for_matching(filename):
    """
    Convert actual Pentacam filenames into their base name
    for matching with the mapping Excel.

    Examples
    --------

    INDEX-LOAD_04-09-2025.csv
        ->
    index-load.csv


    PACHY-LOAD_04-09-2025.csv
        ->
    pachy-load.csv


    PNS_Densito-LOAD_04-09-2025.csv
        ->
    pns_densito-load.csv


    Power_Sag-LOAD_04-09-2025.csv
        ->
    power_sag-load.csv


    PNS_Densito-LOAD.csv
        ->
    pns_densito-load.csv
    """

    filename = Path(filename).name

    # Remove extension
    stem = Path(filename).stem

    # --------------------------------------------------------
    # Remove date at the END
    #
    # Examples:
    #
    # _04-09-2025
    # -04-09-2025
    # _04.09.2025
    # _04_09_2025
    #
    # --------------------------------------------------------

    date_pattern = re.compile(
        r"""
        [_-]?
        \d{1,2}
        [-_.]
        \d{1,2}
        [-_.]
        \d{2,4}
        $
        """,
        re.VERBOSE
    )

    stem = date_pattern.sub(
        "",
        stem
    )


    # --------------------------------------------------------
    # Normalize case
    # --------------------------------------------------------

    stem = stem.strip().lower()

    # Normalize spaces
    stem = re.sub(
        r"\s+",
        " ",
        stem
    )


    return stem + ".csv"


# ============================================================
#              NORMALIZE IDENTIFIER VALUES
# ============================================================

def normalize_text_identifier(value):
    """
    Normalize text identifiers for matching.

    Original values are NOT changed in the final output.
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    # Remove extra spaces
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


# ============================================================
#               NORMALIZE DATE OF BIRTH
# ============================================================

def normalize_dob(value):
    """
    Normalize DOB so that different representations of the
    same date can match.

    Examples:
        12/05/2010
        12-05-2010
        2010-05-12
        12-May-2010

    become:
        2010-05-12
    """

    if pd.isna(value):
        return ""

    value_string = str(value).strip()

    if value_string == "":
        return ""

    # Try pandas date conversion
    try:

        date_value = pd.to_datetime(
            value_string,
            errors="coerce",
            dayfirst=True
        )

        if not pd.isna(date_value):

            return date_value.strftime(
                "%Y-%m-%d"
            )

    except Exception:
        pass


    # If conversion fails, normalize text
    return normalize_text_identifier(
        value
    )


# ============================================================
#              NORMALIZE EXAM EYE
# ============================================================

def normalize_eye(value):
    """
    Standardize eye notation.

    R / Right / RE -> R
    L / Left / LE  -> L
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    value = re.sub(
        r"\s+",
        "",
        value
    )

    eye_mapping = {

        "RIGHT": "R",
        "RE": "R",
        "OD": "R",
        "R": "R",

        "LEFT": "L",
        "LE": "L",
        "OS": "L",
        "L": "L",

        "BOTH": "B",
        "OU": "B",
        "B": "B"
    }

    return eye_mapping.get(
        value,
        value
    )


# ============================================================
#            NORMALIZE ONE IDENTIFIER COLUMN
# ============================================================

def normalize_identifier_column(
    series,
    column_name
):

    if column_name == "D.o.Birth:":

        return series.apply(
            normalize_dob
        )

    elif column_name == "Exam Eye:":

        return series.apply(
            normalize_eye
        )

    else:

        return series.apply(
            normalize_text_identifier
        )


# ============================================================
#                FIND ACTUAL COLUMN
# ============================================================

def find_actual_column(
    dataframe,
    requested_column
):

    requested = normalize_column_name(
        requested_column
    )

    for actual_column in dataframe.columns:

        actual = normalize_column_name(
            actual_column
        )

        if actual == requested:

            return actual_column

    return None


# ============================================================
#                    READ CSV FILE
# ============================================================

def read_csv_file(csv_path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin1",
        "cp1252"
    ]

    for encoding in encodings:

        try:

            dataframe = pd.read_csv(
                csv_path,
                encoding=encoding,
                low_memory=False
            )

            return dataframe

        except UnicodeDecodeError:

            continue

        except Exception as error:

            print(
                f"ERROR reading {csv_path.name}: "
                f"{error}"
            )

            return None


    print(
        f"Could not decode: "
        f"{csv_path.name}"
    )

    return None


# ============================================================
#              READ MAPPING EXCEL
# ============================================================

print("\n")
print("=" * 90)
print("READING MAPPING EXCEL")
print("=" * 90)


mapping_df = pd.read_excel(
    MAPPING_FILE,
    sheet_name=MAPPING_SHEET
)


# Normalize mapping column names

mapping_df.columns = [
    normalize_column_name(column)
    for column in mapping_df.columns
]


print(
    "Mapping columns:"
)

print(
    mapping_df.columns.tolist()
)


# ------------------------------------------------------------
# Check required mapping columns
# ------------------------------------------------------------

required_mapping_columns = [
    "Exact Export File",
    "Exact Column Header"
]


for column in required_mapping_columns:

    if column not in mapping_df.columns:

        raise ValueError(
            f"\nERROR:\n"
            f"Column '{column}' was not found "
            f"in sheet '{MAPPING_SHEET}'.\n\n"
            f"Available columns:\n"
            f"{mapping_df.columns.tolist()}"
        )


# ------------------------------------------------------------
# Remove blank rows
# ------------------------------------------------------------

mapping_df = mapping_df.dropna(
    subset=required_mapping_columns
).copy()


# ------------------------------------------------------------
# Convert to strings
# ------------------------------------------------------------

mapping_df[
    "Exact Export File"
] = (
    mapping_df[
        "Exact Export File"
    ]
    .astype(str)
    .str.strip()
)


mapping_df[
    "Exact Column Header"
] = (
    mapping_df[
        "Exact Column Header"
    ]
    .astype(str)
    .str.strip()
)


# ============================================================
#       CREATE FILE -> COLUMN MAPPING DICTIONARY
# ============================================================

file_column_mapping = (
    mapping_df
    .groupby(
        "Exact Export File"
    )[
        "Exact Column Header"
    ]
    .apply(list)
    .to_dict()
)


print("\n")
print("=" * 90)
print("FILE -> COLUMN MAPPING")
print("=" * 90)


for filename, columns in file_column_mapping.items():

    print(
        f"\n{filename}"
    )

    for column in columns:

        print(
            f"    -> {column}"
        )


# ============================================================
#           PROCESS ONE DATA FOLDER
# ============================================================

def process_data_folder(
    data_folder
):

    print("\n")
    print("#" * 90)
    print(
        f"PROCESSING FOLDER:\n"
        f"{data_folder}"
    )
    print("#" * 90)


    # ========================================================
    # FIND CSV FILES
    # ========================================================

    csv_files = [
        file
        for file in data_folder.glob("*")
        if (
            file.is_file()
            and file.suffix.lower() == ".csv"
        )
    ]


    if not csv_files:

        print(
            "No CSV files found."
        )

        return None


    print(
        f"\nCSV files found: "
        f"{len(csv_files)}"
    )


    # ========================================================
    # CREATE NORMALIZED FILE LOOKUP
    # ========================================================

    csv_lookup = {}


    for csv_file in csv_files:

        normalized_name = (
            normalize_file_for_matching(
                csv_file.name
            )
        )


        if normalized_name in csv_lookup:

            print(
                "\nWARNING:"
            )

            print(
                f"Multiple files match "
                f"'{normalized_name}':"
            )

            print(
                f"  Existing: "
                f"{csv_lookup[normalized_name].name}"
            )

            print(
                f"  New     : "
                f"{csv_file.name}"
            )

            print(
                "The first file will be used."
            )

        else:

            csv_lookup[
                normalized_name
            ] = csv_file


        print(
            f"  {csv_file.name}"
            f"  ->  "
            f"{normalized_name}"
        )


    # ========================================================
    # STORAGE
    # ========================================================

    dataframes = []

    validation_records = []

    identifier_validation_records = []


    # ========================================================
    # PROCESS EVERY REQUIRED CSV
    # ========================================================

    for (
        required_file,
        required_columns
    ) in file_column_mapping.items():


        normalized_required_file = (
            normalize_file_for_matching(
                required_file
            )
        )


        print("\n")
        print(
            f"Required file: "
            f"{required_file}"
        )


        # ====================================================
        # FIND FILE
        # ====================================================

        if (
            normalized_required_file
            not in csv_lookup
        ):

            print(
                "  -> FILE NOT FOUND"
            )


            validation_records.append({

                "Folder":
                    str(data_folder),

                "File":
                    required_file,

                "Status":
                    "Missing File",

                "Column":
                    "",

                "Details":
                    ""

            })

            continue


        csv_path = csv_lookup[
            normalized_required_file
        ]


        print(
            f"  -> MATCHED: "
            f"{csv_path.name}"
        )


        # ====================================================
        # READ CSV
        # ====================================================

        dataframe = read_csv_file(
            csv_path
        )


        if dataframe is None:

            validation_records.append({

                "Folder":
                    str(data_folder),

                "File":
                    csv_path.name,

                "Status":
                    "Read Error",

                "Column":
                    "",

                "Details":
                    ""

            })

            continue


        # ====================================================
        # NORMALIZE CSV COLUMN NAMES
        # ====================================================

        dataframe.columns = [
            normalize_column_name(
                column
            )
            for column in dataframe.columns
        ]


        # ====================================================
        # FIND IDENTIFIER COLUMNS
        # ====================================================

        identifier_actual = {}

        missing_identifier_columns = []


        for identifier in IDENTIFIER_COLUMNS:

            actual_column = (
                find_actual_column(
                    dataframe,
                    identifier
                )
            )


            if actual_column is None:

                missing_identifier_columns.append(
                    identifier
                )

                print(
                    f"  MISSING IDENTIFIER: "
                    f"{identifier}"
                )

            else:

                identifier_actual[
                    identifier
                ] = actual_column


        # ----------------------------------------------------
        # If any identifier is missing, skip file
        # ----------------------------------------------------

        if missing_identifier_columns:

            for identifier in (
                missing_identifier_columns
            ):

                validation_records.append({

                    "Folder":
                        str(data_folder),

                    "File":
                        csv_path.name,

                    "Status":
                        "Missing Identifier",

                    "Column":
                        identifier,

                    "Details":
                        ""

                })


            print(
                "  -> SKIPPED because "
                "identifier columns are missing."
            )

            continue


        # ====================================================
        # FIND REQUESTED DATA COLUMNS
        # ====================================================

        selected_data_columns = []

        missing_data_columns = []


        for requested_column in (
            required_columns
        ):


            # Don't extract identifier columns as
            # parameter columns.
            if requested_column in (
                IDENTIFIER_COLUMNS
            ):

                continue


            actual_column = (
                find_actual_column(
                    dataframe,
                    requested_column
                )
            )


            if actual_column is None:

                missing_data_columns.append(
                    requested_column
                )

                print(
                    f"  MISSING COLUMN: "
                    f"{requested_column}"
                )


                validation_records.append({

                    "Folder":
                        str(data_folder),

                    "File":
                        csv_path.name,

                    "Status":
                        "Missing Column",

                    "Column":
                        requested_column,

                    "Details":
                        ""

                })

            else:

                selected_data_columns.append(
                    actual_column
                )


        # ====================================================
        # CREATE EXTRACTED DATAFRAME
        # ====================================================

        extracted_df = pd.DataFrame()


        # ----------------------------------------------------
        # Add identifiers
        # ----------------------------------------------------

        for identifier in (
            IDENTIFIER_COLUMNS
        ):

            actual_column = (
                identifier_actual[
                    identifier
                ]
            )

            extracted_df[
                identifier
            ] = dataframe[
                actual_column
            ].copy()


        # ----------------------------------------------------
        # Add requested parameters
        # ----------------------------------------------------

        for column in (
            selected_data_columns
        ):

            extracted_df[
                normalize_column_name(
                    column
                )
            ] = dataframe[
                column
            ].copy()


        # ====================================================
        # CREATE NORMALIZED MATCH COLUMNS
        # ====================================================

        for identifier in (
            IDENTIFIER_COLUMNS
        ):

            extracted_df[
                f"_MATCH_{identifier}"
            ] = normalize_identifier_column(
                extracted_df[
                    identifier
                ],
                identifier
            )


        # ====================================================
        # CREATE PRIMARY MATCH KEY
        # ====================================================

        extracted_df[
            "_MATCH_KEY"
        ] = (

            extracted_df[
                "_MATCH_Pat-ID:"
            ].astype(str)

            + "|"

            + extracted_df[
                "_MATCH_D.o.Birth:"
            ].astype(str)

            + "|"

            + extracted_df[
                "_MATCH_Exam Eye:"
            ].astype(str)

        )


        # ====================================================
        # CHECK EMPTY MATCH IDs
        # ====================================================

        empty_key_mask = (
            extracted_df[
                "_MATCH_KEY"
            ]
            .apply(
                lambda x:
                all(
                    part == ""
                    for part in str(x).split("|")
                )
            )
        )


        empty_key_count = (
            empty_key_mask.sum()
        )


        if empty_key_count > 0:

            print(
                f"  WARNING: "
                f"{empty_key_count} records have "
                f"empty primary identifiers."
            )


            validation_records.append({

                "Folder":
                    str(data_folder),

                "File":
                    csv_path.name,

                "Status":
                    "Empty Match Identifier",

                "Column":
                    "Pat-ID / DOB / Eye",

                "Details":
                    f"{empty_key_count} rows"

            })


        # ====================================================
        # CHECK DUPLICATE MATCH KEYS
        # ====================================================

        duplicate_mask = (
            extracted_df[
                "_MATCH_KEY"
            ]
            .duplicated(
                keep=False
            )
        )


        duplicate_count = (
            duplicate_mask.sum()
        )


        if duplicate_count > 0:

            print(
                f"  WARNING: "
                f"{duplicate_count} duplicate "
                f"records found."
            )


            validation_records.append({

                "Folder":
                    str(data_folder),

                "File":
                    csv_path.name,

                "Status":
                    "Duplicate Match Key",

                "Column":
                    "Pat-ID / DOB / Eye",

                "Details":
                    f"{duplicate_count} rows"

            })


        # ====================================================
        # CHECK DUPLICATE DATA COLUMNS
        # ====================================================

        # Data columns only
        parameter_columns = [
            column
            for column in extracted_df.columns
            if (
                column not in IDENTIFIER_COLUMNS
                and not str(
                    column
                ).startswith("_MATCH_")
            )
        ]


        # ----------------------------------------------------
        # Add source prefix if required
        # ----------------------------------------------------

        renamed_columns = {}

        for column in parameter_columns:

            # Check whether this parameter already exists
            # in a previously processed dataframe.
            existing_columns = []

            for existing_df in dataframes:

                existing_columns.extend(
                    existing_df.columns.tolist()
                )


            if column in existing_columns:

                prefix = Path(
                    required_file
                ).stem

                new_name = (
                    f"{prefix}_{column}"
                )

            else:

                new_name = column


            renamed_columns[
                column
            ] = new_name


        extracted_df = (
            extracted_df
            .rename(
                columns=renamed_columns
            )
        )


        # ====================================================
        # REMOVE DUPLICATE MATCH RECORDS
        # ====================================================

        extracted_df = (
            extracted_df
            .drop_duplicates(
                subset="_MATCH_KEY",
                keep="first"
            )
        )


        # ====================================================
        # KEEP ONLY MATCH KEY + PARAMETERS
        # ====================================================

        keep_columns = [
            "_MATCH_KEY"
        ] + [
            column
            for column in extracted_df.columns
            if (
                column not in IDENTIFIER_COLUMNS
                and not str(
                    column
                ).startswith("_MATCH_")
            )
        ]


        extracted_df = extracted_df[
            keep_columns
        ]


        # ====================================================
        # STORE
        # ====================================================

        dataframes.append(
            extracted_df
        )


        print(
            f"  -> Extracted "
            f"{len(extracted_df)} records"
        )

        print(
            f"  -> Extracted "
            f"{len(selected_data_columns)} parameters"
        )


    # ========================================================
    # CHECK IF ANY DATA WAS EXTRACTED
    # ========================================================

    if not dataframes:

        print("\n")
        print(
            "NO USABLE CSV FILES FOUND."
        )

        return None


    # ========================================================
    # MERGE ALL FILES
    # ========================================================

    print("\n")
    print("=" * 90)
    print(
        "MERGING USING:"
    )

    print(
        "Pat-ID + D.o.Birth + Exam Eye"
    )

    print("=" * 90)


    merged_df = dataframes[0].copy()


    for next_df in dataframes[1:]:

        merged_df = pd.merge(

            merged_df,

            next_df,

            on="_MATCH_KEY",

            how="outer",

            suffixes=(
                "",
                "_duplicate"
            )

        )


    # ========================================================
    # RECONSTRUCT IDENTIFIERS
    # ========================================================

    match_parts = (
        merged_df[
            "_MATCH_KEY"
        ]
        .astype(str)
        .str.split(
            "|",
            expand=True
        )
    )


    match_parts.columns = [
        "_MATCH_Pat-ID:",
        "_MATCH_D.o.Birth:",
        "_MATCH_Exam Eye:"
    ]


    # --------------------------------------------------------
    # Identify a source dataframe that contains identifiers
    # --------------------------------------------------------

    identifier_source = None


    for dataframe in dataframes:

        if "_MATCH_KEY" in dataframe.columns:

            identifier_source = dataframe

            break


    # ========================================================
    # CREATE FINAL IDENTIFIER TABLE
    # ========================================================

    # We need the actual original Last Name, First Name,
    # Pat-ID, DOB and Eye values.
    #
    # Use the first available matching record from the
    # source files.

    identifier_table = None


    for source_df in dataframes:

        # Since identifiers were removed from each extracted
        # dataframe after creating the match key, use the
        # original CSV processing logic below by reconstructing
        # from the match key.
        pass


    # ========================================================
    # RECONSTRUCT PRIMARY IDENTIFIERS
    # ========================================================

    final_identifiers = pd.DataFrame()

    final_identifiers[
        "Pat-ID:"
    ] = match_parts[
        "_MATCH_Pat-ID:"
    ]

    final_identifiers[
        "D.o.Birth:"
    ] = match_parts[
        "_MATCH_D.o.Birth:"
    ]

    final_identifiers[
        "Exam Eye:"
    ] = match_parts[
        "_MATCH_Exam Eye:"
    ]


    # --------------------------------------------------------
    # For Last Name and First Name, recover them from the
    # original CSVs using the primary key.
    # --------------------------------------------------------

    name_lookup = {}


    for required_file, required_columns in (
        file_column_mapping.items()
    ):


        normalized_required_file = (
            normalize_file_for_matching(
                required_file
            )
        )


        if normalized_required_file not in csv_lookup:

            continue


        csv_path = csv_lookup[
            normalized_required_file
        ]


        dataframe = read_csv_file(
            csv_path
        )


        if dataframe is None:

            continue


        dataframe.columns = [
            normalize_column_name(
                column
            )
            for column in dataframe.columns
        ]


        # Find identifier columns
        identifier_actual = {}

        identifiers_available = True


        for identifier in IDENTIFIER_COLUMNS:

            actual_column = (
                find_actual_column(
                    dataframe,
                    identifier
                )
            )


            if actual_column is None:

                identifiers_available = False

                break


            identifier_actual[
                identifier
            ] = actual_column


        if not identifiers_available:

            continue


        temp = pd.DataFrame()


        for identifier in (
            IDENTIFIER_COLUMNS
        ):

            temp[
                identifier
            ] = dataframe[
                identifier_actual[
                    identifier
                ]
            ]


        # Create normalized values
        temp[
            "_KEY_PAT"
        ] = normalize_identifier_column(
            temp["Pat-ID:"],
            "Pat-ID:"
        )


        temp[
            "_KEY_DOB"
        ] = normalize_identifier_column(
            temp["D.o.Birth:"],
            "D.o.Birth:"
        )


        temp[
            "_KEY_EYE"
        ] = normalize_identifier_column(
            temp["Exam Eye:"],
            "Exam Eye:"
        )


        temp[
            "_MATCH_KEY"
        ] = (
            temp["_KEY_PAT"].astype(str)
            + "|"
            + temp["_KEY_DOB"].astype(str)
            + "|"
            + temp["_KEY_EYE"].astype(str)
        )


        # Keep first occurrence
        temp = temp.drop_duplicates(
            subset="_MATCH_KEY",
            keep="first"
        )


        # Add to lookup
        for _, row in temp.iterrows():

            key = row[
                "_MATCH_KEY"
            ]


            if key not in name_lookup:

                name_lookup[key] = {

                    "Last Name:":
                        row["Last Name:"],

                    "First Name:":
                        row["First Name:"],

                    "Pat-ID:":
                        row["Pat-ID:"],

                    "D.o.Birth:":
                        row["D.o.Birth:"],

                    "Exam Eye:":
                        row["Exam Eye:"]
                }


    # ========================================================
    # BUILD IDENTIFIER DATAFRAME
    # ========================================================

    identifier_rows = []


    for key in merged_df[
        "_MATCH_KEY"
    ]:

        if key in name_lookup:

            identifier_rows.append(
                name_lookup[key]
            )

        else:

            parts = str(key).split("|")

            identifier_rows.append({

                "Last Name:":
                    "",

                "First Name:":
                    "",

                "Pat-ID:":
                    parts[0]
                    if len(parts) > 0
                    else "",

                "D.o.Birth:":
                    parts[1]
                    if len(parts) > 1
                    else "",

                "Exam Eye:":
                    parts[2]
                    if len(parts) > 2
                    else ""
            })


    identifier_df = pd.DataFrame(
        identifier_rows
    )


    # ========================================================
    # CHECK NAME CONSISTENCY
    # ========================================================

    # Re-read source files and compare names for the same
    # primary identifier.

    name_validation = []


    for required_file, required_columns in (
        file_column_mapping.items()
    ):


        normalized_required_file = (
            normalize_file_for_matching(
                required_file
            )
        )


        if normalized_required_file not in csv_lookup:

            continue


        csv_path = csv_lookup[
            normalized_required_file
        ]


        dataframe = read_csv_file(
            csv_path
        )


        if dataframe is None:

            continue


        dataframe.columns = [
            normalize_column_name(
                column
            )
            for column in dataframe.columns
        ]


        # Find columns
        actual_columns = {}


        for identifier in IDENTIFIER_COLUMNS:

            actual = find_actual_column(
                dataframe,
                identifier
            )


            if actual is None:

                break


            actual_columns[
                identifier
            ] = actual


        if len(actual_columns) != len(
            IDENTIFIER_COLUMNS
        ):

            continue


        temp = pd.DataFrame()


        for identifier in (
            IDENTIFIER_COLUMNS
        ):

            temp[
                identifier
            ] = dataframe[
                actual_columns[
                    identifier
                ]
            ]


        temp[
            "_PAT"
        ] = normalize_identifier_column(
            temp["Pat-ID:"],
            "Pat-ID:"
        )


        temp[
            "_DOB"
        ] = normalize_identifier_column(
            temp["D.o.Birth:"],
            "D.o.Birth:"
        )


        temp[
            "_EYE"
        ] = normalize_identifier_column(
            temp["Exam Eye:"],
            "Exam Eye:"
        )


        temp[
            "_KEY"
        ] = (
            temp["_PAT"].astype(str)
            + "|"
            + temp["_DOB"].astype(str)
            + "|"
            + temp["_EYE"].astype(str)
        )


        for key, group in temp.groupby(
            "_KEY"
        ):

            last_names = set(
                group[
                    "Last Name:"
                ]
                .dropna()
                .apply(
                    normalize_text_identifier
                )
            )


            first_names = set(
                group[
                    "First Name:"
                ]
                .dropna()
                .apply(
                    normalize_text_identifier
                )
            )


            if len(last_names) > 1:

                name_validation.append({

                    "File":
                        csv_path.name,

                    "Match Key":
                        key,

                    "Field":
                        "Last Name:",

                    "Values":
                        " | ".join(
                            sorted(
                                last_names
                            )
                        ),

                    "Status":
                        "Mismatch"
                })


            if len(first_names) > 1:

                name_validation.append({

                    "File":
                        csv_path.name,

                    "Match Key":
                        key,

                    "Field":
                        "First Name:",

                    "Values":
                        " | ".join(
                            sorted(
                                first_names
                            )
                        ),

                    "Status":
                        "Mismatch"
                })


    # ========================================================
    # REMOVE INTERNAL MATCH COLUMNS
    # ========================================================

    parameter_df = merged_df.drop(
        columns=[
            "_MATCH_KEY"
        ]
    )


    # --------------------------------------------------------
    # Remove any duplicate columns created by merge
    # --------------------------------------------------------

    duplicate_columns = [
        column
        for column in parameter_df.columns
        if str(
            column
        ).endswith(
            "_duplicate"
        )
    ]


    if duplicate_columns:

        parameter_df = parameter_df.drop(
            columns=duplicate_columns
        )


    # ========================================================
    # FINAL DATAFRAME
    # ========================================================

    final_df = pd.concat(
        [
            identifier_df,
            parameter_df
        ],
        axis=1
    )


    # ========================================================
    # REMOVE COMPLETELY EMPTY ROWS
    # ========================================================

    final_df = final_df.dropna(
        how="all"
    )


    # ========================================================
    # SAVE MERGED EXCEL
    # ========================================================

    output_path = (
        data_folder /
        OUTPUT_FILE
    )


    final_df.to_excel(
        output_path,
        index=False
    )


    # ========================================================
    # SAVE VALIDATION REPORT
    # ========================================================

    validation_path = (
        data_folder /
        VALIDATION_FILE
    )


    with pd.ExcelWriter(
        validation_path,
        engine="openpyxl"
    ) as writer:


        # -----------------------------------------------
        # Missing files / columns
        # -----------------------------------------------

        if validation_records:

            validation_df = pd.DataFrame(
                validation_records
            )

        else:

            validation_df = pd.DataFrame({

                "Status":
                    ["No missing files or columns"]

            })


        validation_df.to_excel(
            writer,
            sheet_name="Files_Columns",
            index=False
        )


        # -----------------------------------------------
        # Name mismatches
        # -----------------------------------------------

        if name_validation:

            name_validation_df = pd.DataFrame(
                name_validation
            )

        else:

            name_validation_df = pd.DataFrame({

                "Status":
                    ["No name mismatches found"]

            })


        name_validation_df.to_excel(
            writer,
            sheet_name="Name_Check",
            index=False
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n")
    print("=" * 90)
    print("MERGE COMPLETED")
    print("=" * 90)

    print(
        f"Rows       : "
        f"{len(final_df)}"
    )

    print(
        f"Columns    : "
        f"{len(final_df.columns)}"
    )

    print(
        f"Output     : "
        f"{output_path}"
    )

    print(
        f"Validation : "
        f"{validation_path}"
    )

    print("=" * 90)


    return final_df


# ============================================================
#              FIND ALL DATA FOLDERS
# ============================================================

print("\n")
print("=" * 90)
print("SEARCHING FOR DATA FOLDERS")
print("=" * 90)


data_folders = []


for folder in MAIN_FOLDER.rglob("*"):

    if not folder.is_dir():

        continue


    # Only consider folders that directly contain CSV files

    csv_files = [
        file
        for file in folder.glob("*")
        if (
            file.is_file()
            and file.suffix.lower() == ".csv"
        )
    ]


    if csv_files:

        data_folders.append(
            folder
        )


print(
    f"\nTotal folders containing CSVs: "
    f"{len(data_folders)}"
)


# ============================================================
#              PROCESS ALL DATA FOLDERS
# ============================================================

successful = 0
failed = 0


for i, folder in enumerate(
    data_folders,
    start=1
):


    print("\n")
    print(
        f"[{i}/{len(data_folders)}]"
    )


    try:

        result = process_data_folder(
            folder
        )


        if result is not None:

            successful += 1

        else:

            failed += 1


    except Exception as error:

        failed += 1


        print("\n")
        print(
            "ERROR PROCESSING FOLDER:"
        )

        print(
            folder
        )

        print(
            f"ERROR: {error}"
        )


# ============================================================
#                       FINAL SUMMARY
# ============================================================

print("\n")
print("#" * 90)
print("ALL PROCESSING COMPLETED")
print("#" * 90)

print(
    f"Folders containing CSVs : "
    f"{len(data_folders)}"
)

print(
    f"Successfully merged     : "
    f"{successful}"
)

print(
    f"Failed                  : "
    f"{failed}"
)

print("#" * 90)



READING MAPPING EXCEL
Mapping columns:
['Exact Export File', 'Exact Column Header']


FILE -> COLUMN MAPPING

BADisplay-LOAD.CSV
    -> Pachy Prog Index Min.:
    -> Pachy Prog Index Max.:
    -> Pachy Prog Index Avg.:
    -> ART Max.:
    -> ART Min.:
    -> ART Avg.:
    -> Dist. Apex-Thin.Loc. [mm]:
    -> BAD Df:
    -> BAD Db:
    -> BAD Dp:
    -> BAD Dt:
    -> BAD Dam:
    -> BAD D:

CHAMBER-LOAD.CSV
    -> C.Volume:
    -> C.Angle:
    -> Pupil:

CorneoScleral-LOAD.CSV
    -> HWTW:

INDEX-LOAD.CSV
    -> K Max (Front):
    -> K Max X (Front):
    -> K Max Y (Front):

PACHY-LOAD.CSV
    -> AC Depth


SEARCHING FOR DATA FOLDERS

Total folders containing CSVs: 1


[1/1]


##########################################################################################
PROCESSING FOLDER:
C:\xx\ADITHYA TALLENT
##########################################################################################

CSV files found: 15
  BADisplay-LOAD_04-09-2025.CSV  ->  badisplay-load.csv
  CHAMBER-L